## BASIC RAG PIPELINE

#### STEP 1: Fetch the Documents from arxiv

In [30]:
from pathlib import Path
import arxiv
import requests

data_folder = Path("data/pdfs")
data_folder.mkdir(parents=True, exist_ok=True)

query = "retrieval augmented generation"
number_of_pages = 5

search = arxiv.Search(
    query=query,
    max_results=number_of_pages,
    sort_by=arxiv.SortCriterion.Relevance,
)

client = arxiv.Client(page_size=number_of_pages)

pdf_paths = []

for file_number, paper in enumerate(client.results(search), start=1):
    file_name = f"Paper_{file_number:02d}.pdf"
    pdf_path = data_folder / file_name
    pdf_path.write_bytes(requests.get(paper.pdf_url).content)
    pdf_paths.append(pdf_path)
    print(f"Downloaded: {paper.title} -> {pdf_path}")
print(f"\nSuccessfully downloaded {len(pdf_paths)} PDF files.\n")

Downloaded: AR-RAG: Autoregressive Retrieval Augmentation for Image Generation -> data\pdfs\Paper_01.pdf
Downloaded: Intelligent Interaction Strategies for Context-Aware Cognitive Augmentation -> data\pdfs\Paper_02.pdf
Downloaded: Factually: Exploring Wearable Fact-Checking for Augmented Truth Discernment -> data\pdfs\Paper_03.pdf
Downloaded: Designing AI Systems that Augment Human Performed vs. Demonstrated Critical Thinking -> data\pdfs\Paper_04.pdf
Downloaded: Automated Literature Review Using NLP Techniques and LLM-Based Retrieval-Augmented Generation -> data\pdfs\Paper_05.pdf

Successfully downloaded 5 PDF files.



#### Step 2: Load the Documents

In [31]:
from langchain_community.document_loaders import DirectoryLoader, PyPDFLoader

loader = DirectoryLoader(
    path="./data/pdfs",
    glob="*.pdf",
    loader_cls=PyPDFLoader,
)

loaded_documents = loader.load()
loaded_documents

[Document(metadata={'producer': 'pikepdf 8.15.1', 'creator': 'arXiv GenPDF (tex2pdf:)', 'creationdate': '', 'author': 'Jingyuan Qi; Zhiyang Xu; Qifan Wang; Lifu Huang', 'doi': 'https://doi.org/10.48550/arXiv.2506.06962', 'license': 'http://creativecommons.org/licenses/by/4.0/', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.25 (TeX Live 2023) kpathsea version 6.3.5', 'title': 'AR-RAG: Autoregressive Retrieval Augmentation for Image Generation', 'trapped': '/False', 'arxivid': 'https://arxiv.org/abs/2506.06962v3', 'source': 'data\\pdfs\\Paper_01.pdf', 'total_pages': 18, 'page': 0, 'page_label': '1'}, page_content='arXiv:2506.06962v3  [cs.CV]  14 Jun 2025\nAR-RAG : Autoregressive Retrieval Augmentation for\nImage Generation\nJingyuan Qi* 1 Zhiyang Xu* 1 Qifan Wang2 Lifu Huang3\n1Virginia Tech 2Meta 3 UC Davis\njingyq1@vt.edu\n(a)\tVanilla\tImage\tGeneration\nPrompt\n(c)\tPatch-based\tAutoregressive\tRetrieval\tAugmentation\t(Ours)\n...\nAugmentation\nGeneration\nPrompt\

#### Step 3: Chunking

In [32]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=700,
    chunk_overlap=80,
    separators=["\n\n", "\n", " "],
)

chunks = splitter.split_documents(loaded_documents)
len(chunks)

272

#### Step 4: Embeddings

In [33]:
from langchain_openai import OpenAIEmbeddings

embeddings_model = OpenAIEmbeddings(model="text-embedding-3-small")

In [37]:
# Embeddings (with caching)
from langchain_classic.storage import LocalFileStore
from langchain_classic.embeddings.cache import CacheBackedEmbeddings

store = LocalFileStore("./embedding_cache/documents")

embeddings_with_cache = CacheBackedEmbeddings.from_bytes_store(
    underlying_embeddings=embeddings_model,
    document_embedding_cache=store,
    namespace="BasicRAG"
)

#### Step 5: Vector Store - Chromadb

In [39]:
from langchain_chroma import Chroma

import os

if os.path.exists("./chroma_db") and os.listdir("./chroma_db"):
    vectorstore = Chroma(
        persist_directory="./chroma_db",
        embedding_function=embeddings_with_cache,
        collection_name="BasicRAG"
    )
    print("Loaded existing store:", vectorstore._collection.count())
else:
    vectorstore = Chroma.from_documents(
        documents=chunks,
        embedding=embeddings_with_cache,
        persist_directory="./chroma_db",
        collection_name="BasicRAG"
    )
    print("Loaded", vectorstore._collection.count())

Loaded existing store: 272


#### Step 6: Get context and Question

Context <- Retriever

Question <- RunnablePassThrough

In [46]:
from langchain_core.runnables import RunnablePassthrough, RunnableParallel

retriever = vectorstore.as_retriever(
    search_type="mmr",
    search_kwargs={"k": 5, "fetch_k": 20, "lambda_mult": 0.5}
)

runnables = RunnableParallel(
    question= RunnablePassthrough(),
    context= retriever | (lambda docs: "\n\n".join(doc.page_content for doc in docs)),
)

#### Step 7: Prompt Template

In [47]:
from langchain_core.prompts import ChatPromptTemplate

prompt_template = ChatPromptTemplate.from_template(
"""
Answer the question based on the following context :
{context}

Question: {question}
Answer:

Make sure to answer in a concise manner,
and if you don't know the answer, just say "I don't know"
""")

#### Step 8: LLM Model and Parser

In [48]:
from langchain_openai import ChatOpenAI
from langchain_core.output_parsers import StrOutputParser

llm_model = ChatOpenAI(model="gpt-4o-mini", temperature=0.2)

parser = StrOutputParser()

#### Step 9: Chaining

In [49]:
chain = runnables | prompt_template | llm_model | parser

In [50]:
questions = [
    "How can AI systems adapt to human cognitive states?",
    "What are the limitations of retrieval-augmented generation?",
    "How does context-awareness improve AI system performance?",
]

for i, question in enumerate(questions, start=1):
    answer = chain.invoke(question)
    print(f"Question {i} : {question}\n")
    print(f"Answer: {answer}\n")
    print("-------------------------------------")


Question 1 : How can AI systems adapt to human cognitive states?

Answer: AI systems can adapt to human cognitive states by recognizing whether a user is engaging in exploratory or structured information processing and adjusting their interventions accordingly. Additionally, they can integrate multi-modal awareness by considering text, images, movement patterns, and behavioral cues to enhance contextualized knowledge. AI can also employ user embeddings to personalize responses based on interaction history and cognitive style, while utilizing knowledge graphs to represent user information and preferences for more tailored support.

-------------------------------------
Question 2 : What are the limitations of retrieval-augmented generation?

Answer: The limitations of retrieval-augmented generation (RAG) include its underdeveloped application in image and multimodal generation, potential degradation in instruction-following capability due to overcopying behavior, and biases in the gener